<a href="https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/12_grid_world_dqn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grid World Reinforcement Learning - Deep Q-Network (DQN)

This notebook demonstrates **Deep Q-Network (DQN)** applied to our grid world problem. Students will learn:

1. **Deep Q-Network**: Using neural networks to approximate Q-values
2. **Experience Replay**: Learning from stored experiences to break correlation
3. **Target Network**: Stabilizing training with separate target networks
4. **Epsilon Decay**: Gradually reducing exploration as learning progresses
5. **Deep Learning Integration**: Combining RL with neural network function approximation

## Learning Objectives

- Understand how neural networks can approximate value functions
- Learn the importance of experience replay for stable training
- Implement target networks for improved convergence
- Visualize deep learning-based policy evolution
- Compare DQN performance with tabular Q-Learning
- See how DQN scales to larger state spaces

Let's start by setting up our environment and DQN algorithm!

In [ ]:
# Import required libraries
import numpy as np
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import time
from enum import Enum
from typing import Tuple, List, Optional, Dict, Deque
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Libraries imported successfully!")
print("Ready to create our DQN algorithm.")
print("Note: Using PyTorch for neural network implementation.")

Using device: cpu
Libraries imported successfully!
Ready to create our DQN algorithm.
Note: Using PyTorch for neural network implementation.


## 1. Environment Setup (Extended from Previous Notebooks)

We'll use the same GridWorld environment but extend it to work better with neural networks by providing state representations as vectors.

In [ ]:
# Define the cell types and actions (same as before)
class CellType(Enum):
    EMPTY = 0
    OBSTACLE = 1
    GOAL = 2
    PENALTY = 3
    AGENT = 4

class Action(Enum):
    UP = 0
    DOWN = 1
    LEFT = 2
    RIGHT = 3

# Enhanced GridWorld Environment Class for DQN
class GridWorldDQN:
    def __init__(self, width=5, height=5):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)
        self.agent_pos = [0, 0]  # [row, col]
        self.goal_pos = [height-1, width-1]
        self.start_pos = [0, 0]
        
        # Rewards
        self.step_reward = -0.1  # Small penalty for each step
        self.goal_reward = 10.0  # Large reward for reaching goal
        self.obstacle_penalty = -1.0  # Penalty for hitting obstacle
        self.penalty_reward = -5.0  # Large penalty for penalty cells
        
        # State space size for neural network
        self.state_size = width * height  # Flattened grid representation
        self.action_size = len(Action)
        
        # Initialize grid
        self._setup_default_grid()
        
    def _setup_default_grid(self):
        """Setup a default grid with some obstacles and penalties"""
        # Clear the grid
        self.grid.fill(CellType.EMPTY.value)
        
        # Add some obstacles
        if self.width >= 5 and self.height >= 5:
            self.grid[1, 2] = CellType.OBSTACLE.value
            self.grid[2, 2] = CellType.OBSTACLE.value
            self.grid[3, 1] = CellType.OBSTACLE.value
            
            # Add a penalty cell
            self.grid[2, 3] = CellType.PENALTY.value
        
        # Set goal
        self.grid[self.goal_pos[0], self.goal_pos[1]] = CellType.GOAL.value
        
    def reset(self):
        """Reset the agent to starting position"""
        self.agent_pos = self.start_pos.copy()
        return self.get_state_vector()
    
    def get_state(self):
        """Get current state as tuple (row, col)"""
        return tuple(self.agent_pos)
    
    def get_state_vector(self):
        """Get state as a vector for neural network input"""
        # Create a one-hot encoding of agent position
        state_vector = np.zeros(self.state_size)
        agent_idx = self.agent_pos[0] * self.width + self.agent_pos[1]
        state_vector[agent_idx] = 1.0
        
        # Optionally add grid information (obstacles, goals, etc.)
        # For now, we'll use simple position encoding
        return state_vector
    
    def get_state_with_context(self):
        """Get enhanced state representation including local environment"""
        # Position encoding
        pos_encoding = np.zeros(self.state_size)
        agent_idx = self.agent_pos[0] * self.width + self.agent_pos[1]
        pos_encoding[agent_idx] = 1.0
        
        # Grid encoding (flattened)
        grid_encoding = self.grid.flatten() / 4.0  # Normalize cell types
        
        # Combine both encodings
        state_vector = np.concatenate([pos_encoding, grid_encoding])
        return state_vector
    
    def is_valid_action(self, action):
        """Check if action is valid from current position"""
        new_pos = self._get_new_position(action)
        return self._is_valid_position(new_pos)
    
    def _get_new_position(self, action):
        """Calculate new position after taking action"""
        row, col = self.agent_pos
        
        if action == Action.UP:
            return [row - 1, col]
        elif action == Action.DOWN:
            return [row + 1, col]
        elif action == Action.LEFT:
            return [row, col - 1]
        elif action == Action.RIGHT:
            return [row, col + 1]
        else:
            return [row, col]  # Invalid action, stay in place
    
    def _is_valid_position(self, pos):
        """Check if position is within bounds and not an obstacle"""
        row, col = pos
        
        # Check bounds
        if row < 0 or row >= self.height or col < 0 or col >= self.width:
            return False
        
        # Check if it's an obstacle
        if self.grid[row, col] == CellType.OBSTACLE.value:
            return False
        
        return True
    
    def step(self, action):
        """Take a step in the environment"""
        # Convert action index to Action enum if needed
        if isinstance(action, int):
            action = Action(action)
            
        # Calculate new position
        new_pos = self._get_new_position(action)
        
        # Check if move is valid
        if not self._is_valid_position(new_pos):
            # Invalid move - stay in place and get penalty
            reward = self.obstacle_penalty
            done = False
        else:
            # Valid move - update position
            self.agent_pos = new_pos
            
            # Calculate reward based on cell type
            cell_type = self.grid[new_pos[0], new_pos[1]]
            
            if cell_type == CellType.GOAL.value:
                reward = self.goal_reward
                done = True
            elif cell_type == CellType.PENALTY.value:
                reward = self.penalty_reward
                done = False
            else:  # Empty cell
                reward = self.step_reward
                done = False
        
        # Return state vector, reward, done, info
        return self.get_state_vector(), reward, done, {}
    
    def get_possible_actions(self):
        """Get all possible actions from current state"""
        possible_actions = []
        for action in Action:
            if self.is_valid_action(action):
                possible_actions.append(action.value)
        return possible_actions

print("Enhanced GridWorld environment created successfully!")
print("✓ Neural network-compatible state representation")
print("✓ Vector-based state encoding")
print("✓ Same core mechanics as previous notebooks")

Enhanced GridWorld environment created successfully!
✓ Neural network-compatible state representation
✓ Vector-based state encoding
✓ Same core mechanics as previous notebooks


## 2. Deep Q-Network Architecture

The core of DQN is a neural network that approximates the Q-function Q(s,a). Let's implement a flexible network architecture.

In [ ]:
class DQNNetwork(nn.Module):
    """Deep Q-Network for approximating Q-values"""
    
    def __init__(self, state_size, action_size, hidden_sizes=[64, 64]):
        super(DQNNetwork, self).__init__()
        self.state_size = state_size
        self.action_size = action_size
        
        # Build the network layers
        layers = []
        prev_size = state_size
        
        # Hidden layers
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU()
            ])
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, action_size))
        
        # Create the sequential network
        self.network = nn.Sequential(*layers)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Initialize network weights"""
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            module.bias.data.fill_(0.01)
    
    def forward(self, state):
        """Forward pass through the network"""
        return self.network(state)
    
    def get_q_values(self, state):
        """Get Q-values for all actions given a state"""
        if not isinstance(state, torch.Tensor):
            state = torch.FloatTensor(state).to(device)
        
        # Add batch dimension if needed
        if len(state.shape) == 1:
            state = state.unsqueeze(0)
        
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.cpu().numpy()
    
    def get_action(self, state, epsilon=0.0):
        """Get action using epsilon-greedy policy"""
        if random.random() < epsilon:
            return random.randint(0, self.action_size - 1)
        else:
            q_values = self.get_q_values(state)
            return np.argmax(q_values[0])

print("DQN Network architecture implemented!")
print("✓ Flexible hidden layer configuration")
print("✓ Xavier weight initialization")
print("✓ Epsilon-greedy action selection")
print("✓ GPU support ready")

DQN Network architecture implemented!
✓ Flexible hidden layer configuration
✓ Xavier weight initialization
✓ Epsilon-greedy action selection
✓ GPU support ready


## 3. Experience Replay Buffer

Experience replay is crucial for DQN's success. It stores experiences and samples them randomly to break temporal correlations.

In [ ]:
class ReplayBuffer:
    """Experience replay buffer for storing and sampling experiences"""
    
    def __init__(self, capacity=10000):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
        self.position = 0
    
    def push(self, state, action, reward, next_state, done):
        """Save an experience"""
        experience = (state, action, reward, next_state, done)
        self.buffer.append(experience)
    
    def sample(self, batch_size):
        """Sample a batch of experiences"""
        if len(self.buffer) < batch_size:
            batch_size = len(self.buffer)
        
        batch = random.sample(self.buffer, batch_size)
        
        # Unpack the batch
        states = np.array([e[0] for e in batch])
        actions = np.array([e[1] for e in batch])
        rewards = np.array([e[2] for e in batch])
        next_states = np.array([e[3] for e in batch])
        dones = np.array([e[4] for e in batch])
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.buffer)
    
    def clear(self):
        """Clear the buffer"""
        self.buffer.clear()
    
    def get_statistics(self):
        """Get buffer statistics"""
        if len(self.buffer) == 0:
            return {"size": 0, "avg_reward": 0, "capacity_used": 0}
        
        rewards = [exp[2] for exp in self.buffer]
        return {
            "size": len(self.buffer),
            "avg_reward": np.mean(rewards),
            "capacity_used": len(self.buffer) / self.capacity * 100
        }

print("Experience Replay Buffer implemented!")
print("✓ Circular buffer with configurable capacity")
print("✓ Random sampling for training")
print("✓ Buffer statistics tracking")
print("✓ Memory efficient storage")

Experience Replay Buffer implemented!
✓ Circular buffer with configurable capacity
✓ Random sampling for training
✓ Buffer statistics tracking
✓ Memory efficient storage


---

# Part 4: DQN Agent Implementation

## 4. Deep Q-Network Agent

### Equations

**DQN Loss Function:**
```
L(θ) = E[(r + γ max_a' Q(s',a'; θ⁻) - Q(s,a; θ))²]
```

**Key Components:**
- **θ**: Parameters of the main network
- **θ⁻**: Parameters of the target network (updated periodically)
- **Experience Replay**: Random sampling from replay buffer
- **Epsilon Decay**: Gradually reduce exploration over time

Let's implement the complete DQN agent!

In [ ]:
class DQNAgent:
    """Deep Q-Network Agent"""
    
    def __init__(self, env, lr=0.001, gamma=0.99, epsilon=1.0, epsilon_min=0.01, 
                 epsilon_decay=0.995, buffer_size=10000, batch_size=32, 
                 target_update=100, hidden_sizes=[64, 64]):
        
        self.env = env
        self.state_size = env.state_size
        self.action_size = env.action_size
        
        # Hyperparameters
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update = target_update
        
        # Networks
        self.q_network = DQNNetwork(self.state_size, self.action_size, hidden_sizes).to(device)
        self.target_network = DQNNetwork(self.state_size, self.action_size, hidden_sizes).to(device)
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        
        # Initialize target network with same weights
        self.update_target_network()
        
        # Experience replay
        self.replay_buffer = ReplayBuffer(buffer_size)
        
        # Training statistics
        self.episode_rewards = []
        self.episode_steps = []
        self.episode_count = 0
        self.total_steps = 0
        self.losses = []
        
    def update_target_network(self):
        """Copy weights from main network to target network"""
        self.target_network.load_state_dict(self.q_network.state_dict())
    
    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay buffer"""
        self.replay_buffer.push(state, action, reward, next_state, done)
    
    def act(self, state, training=True):
        """Choose action using epsilon-greedy policy"""
        if training and random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)
        else:
            return self.q_network.get_action(state)
    
    def replay(self):
        """Train the network on a batch of experiences"""
        if len(self.replay_buffer) < self.batch_size:
            return None
        
        # Sample batch from replay buffer
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        # Convert to tensors
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.BoolTensor(dones).to(device)
        
        # Current Q values
        current_q_values = self.q_network(states).gather(1, actions.unsqueeze(1))
        
        # Next Q values from target network
        next_q_values = self.target_network(next_states).max(1)[0].detach()
        target_q_values = rewards + (self.gamma * next_q_values * ~dones)
        
        # Compute loss
        loss = F.mse_loss(current_q_values.squeeze(), target_q_values)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        return loss.item()
    
    def run_episode(self, max_steps=100, training=True):
        """Run one episode"""
        state = self.env.reset()
        total_reward = 0
        steps = 0
        episode_experiences = []
        
        for step in range(max_steps):
            # Choose action
            action = self.act(state, training)
            
            # Take action
            next_state, reward, done, _ = self.env.step(action)
            
            # Store experience
            if training:
                self.remember(state, action, reward, next_state, done)
                episode_experiences.append({
                    'state': state.copy(),
                    'action': action,
                    'reward': reward,
                    'next_state': next_state.copy(),
                    'done': done
                })
            
            total_reward += reward
            steps += 1
            state = next_state
            self.total_steps += 1
            
            # Train the network
            loss = None
            if training and len(self.replay_buffer) >= self.batch_size:
                loss = self.replay()
                if loss is not None:
                    self.losses.append(loss)
            
            # Update target network
            if training and self.total_steps % self.target_update == 0:
                self.update_target_network()
            
            if done:
                break
        
        # Store episode statistics
        if training:
            self.episode_rewards.append(total_reward)
            self.episode_steps.append(steps)
            self.episode_count += 1
        
        return episode_experiences, total_reward, steps
    
    def train(self, num_episodes=500, show_progress=True):
        """Train the DQN agent"""
        for episode in range(num_episodes):
            experiences, reward, steps = self.run_episode()
            
            if show_progress and (episode + 1) % 50 == 0:
                avg_reward = np.mean(self.episode_rewards[-50:])
                avg_loss = np.mean(self.losses[-100:]) if self.losses else 0
                print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.2f}, "
                      f"Epsilon = {self.epsilon:.3f}, Avg Loss = {avg_loss:.4f}")
    
    def test_policy(self, max_steps=50):
        """Test the learned policy (no exploration, no training)"""
        state = self.env.reset()
        path = [self.env.get_state()]
        total_reward = 0
        
        for step in range(max_steps):
            action = self.act(state, training=False)
            next_state, reward, done, _ = self.env.step(action)
            
            path.append(self.env.get_state())
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        return path, total_reward, len(path) - 1
    
    def get_q_values_grid(self):
        """Get Q-values for all positions in the grid"""
        q_grid = np.zeros((self.env.height, self.env.width, self.action_size))
        
        for i in range(self.env.height):
            for j in range(self.env.width):
                # Create state vector for this position
                state_vector = np.zeros(self.env.state_size)
                pos_idx = i * self.env.width + j
                state_vector[pos_idx] = 1.0
                
                # Get Q-values from network
                q_values = self.q_network.get_q_values(state_vector)[0]
                q_grid[i, j] = q_values
        
        return q_grid

print("DQN Agent implemented!")
print("✓ Neural network Q-function approximation")
print("✓ Experience replay integration")
print("✓ Target network stabilization")
print("✓ Epsilon decay schedule")
print("✓ Training and testing functionality")

DQN Agent implemented!
✓ Neural network Q-function approximation
✓ Experience replay integration
✓ Target network stabilization
✓ Epsilon decay schedule
✓ Training and testing functionality


## 5. Visualization System for DQN

We'll create visualizations to show how the neural network learns Q-values and policies over time.

In [ ]:
class DQNVisualizer:
    def __init__(self, env):
        self.env = env
        
        # Define symbols for different cell types
        self.symbols = {
            CellType.EMPTY.value: '⬜',
            CellType.OBSTACLE.value: '🚫',
            CellType.GOAL.value: '🎯',
            CellType.PENALTY.value: '🔥'
        }
        
        self.agent_symbol = '🤖'
        self.action_symbols = {
            Action.UP.value: '↑',
            Action.DOWN.value: '↓', 
            Action.LEFT.value: '←',
            Action.RIGHT.value: '→'
        }
        
    def create_grid_html(self, show_agent=True):
        """Create HTML representation of the grid"""
        html = "<div style='font-family: monospace; font-size: 24px; line-height: 1.2;'>"
        html += "<h3 style='text-align: center; margin: 10px 0;'>Grid World Environment</h3>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        for i in range(self.env.height):
            html += "<div style='display: flex;'>"
            for j in range(self.env.width):
                # Check if agent is at this position
                if show_agent and [i, j] == self.env.agent_pos:
                    symbol = self.agent_symbol
                    bg_color = '#E3F2FD'  # Light blue background for agent
                else:
                    cell_type = self.env.grid[i, j]
                    symbol = self.symbols.get(cell_type, '⬜')
                    bg_color = '#F5F5F5'  # Light gray background
                
                html += f"<div style='width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color};'>{symbol}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def create_q_values_html(self, q_grid, title="Neural Network Q-Values"):
        """Create HTML representation of Q-values from neural network"""
        html = "<div style='font-family: monospace; font-size: 10px; line-height: 1.1;'>"
        html += f"<h4 style='text-align: center; margin: 10px 0;'>{title}</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        # Normalize Q-values for color coding
        max_q = np.max(q_grid) if np.max(q_grid) > 0 else 1
        min_q = np.min(q_grid) if np.min(q_grid) < 0 else -1
        
        for i in range(self.env.height):
            html += "<div style='display: flex;'>"
            for j in range(self.env.width):
                if self.env.grid[i, j] == CellType.OBSTACLE.value:
                    # Show obstacle
                    html += "<div style='width: 80px; height: 60px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: #000000; color: white;'>🚫</div>"
                else:
                    # Show Q-values for this state
                    html += "<div style='width: 80px; height: 60px; border: 1px solid #ccc; background-color: #f9f9f9; position: relative;'>"
                    
                    # Add Q-values for each action (arranged in cross pattern)
                    for action_idx in range(4):
                        q_val = q_grid[i, j, action_idx]
                        
                        # Color based on Q-value
                        if q_val > 0:
                            intensity = min(q_val / max_q, 1.0) * 0.8
                            color = f'rgba(0, 255, 0, {intensity})'
                        elif q_val < 0:
                            intensity = min(abs(q_val) / abs(min_q), 1.0) * 0.8
                            color = f'rgba(255, 0, 0, {intensity})'
                        else:
                            color = 'rgba(128, 128, 128, 0.1)'
                        
                        # Position based on action
                        if action_idx == Action.UP.value:
                            style = 'top: 0px; left: 25px; width: 30px; height: 15px;'
                        elif action_idx == Action.DOWN.value:
                            style = 'bottom: 0px; left: 25px; width: 30px; height: 15px;'
                        elif action_idx == Action.LEFT.value:
                            style = 'top: 22px; left: 0px; width: 25px; height: 16px;'
                        else:  # RIGHT
                            style = 'top: 22px; right: 0px; width: 25px; height: 16px;'
                        
                        html += f"<div style='position: absolute; {style} background-color: {color}; border: 1px solid #ddd; display: flex; align-items: center; justify-content: center; font-size: 8px; font-weight: bold;'>{q_val:.1f}</div>"
                    
                    html += "</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def create_policy_html(self, q_grid, title="Neural Network Policy"):
        """Create policy visualization from Q-values"""
        html = "<div style='font-family: monospace; font-size: 20px; line-height: 1.2;'>"
        html += f"<h4 style='text-align: center; margin: 10px 0;'>{title}</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        for i in range(self.env.height):
            html += "<div style='display: flex;'>"
            for j in range(self.env.width):
                if self.env.grid[i, j] == CellType.OBSTACLE.value:
                    symbol = '🚫'
                    bg_color = '#000000'
                elif self.env.grid[i, j] == CellType.GOAL.value:
                    symbol = '🎯'
                    bg_color = '#FFD700'
                elif self.env.grid[i, j] == CellType.PENALTY.value:
                    symbol = '🔥'
                    bg_color = '#FF6B6B'
                else:
                    # Find best action from Q-values
                    best_action_idx = np.argmax(q_grid[i, j])
                    symbol = self.action_symbols[best_action_idx]
                    bg_color = '#E8F5E8'
                
                html += f"<div style='width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color};'>{symbol}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def create_training_progress_html(self, agent):
        """Create training progress visualization"""
        if len(agent.episode_rewards) == 0:
            return "<p>No training data available yet.</p>"
        
        # Calculate moving averages
        window = min(50, len(agent.episode_rewards))
        recent_rewards = agent.episode_rewards[-window:]
        avg_reward = np.mean(recent_rewards)
        
        recent_losses = agent.losses[-100:] if agent.losses else [0]
        avg_loss = np.mean(recent_losses)
        
        buffer_stats = agent.replay_buffer.get_statistics()
        
        html = f"""
        <div style='padding: 10px; background-color: #f9f9f9; margin: 10px 0; border-radius: 5px;'>
            <h4>🧠 DQN Training Progress</h4>
            <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px;'>
                <div>
                    <p><strong>Episodes:</strong> {agent.episode_count}</p>
                    <p><strong>Total Steps:</strong> {agent.total_steps:,}</p>
                    <p><strong>Average Reward (last {window}):</strong> {avg_reward:.2f}</p>
                    <p><strong>Current Epsilon:</strong> {agent.epsilon:.3f}</p>
                </div>
                <div>
                    <p><strong>Average Loss (last 100):</strong> {avg_loss:.4f}</p>
                    <p><strong>Buffer Size:</strong> {buffer_stats['size']:,}</p>
                    <p><strong>Buffer Capacity Used:</strong> {buffer_stats['capacity_used']:.1f}%</p>
                    <p><strong>Network Updates:</strong> {len(agent.losses):,}</p>
                </div>
            </div>
        </div>
        """
        
        return html

print("DQN Visualizer created successfully!")
print("✓ Neural network Q-value visualization")
print("✓ Policy extraction from learned Q-values")
print("✓ Training progress monitoring")
print("✓ Experience buffer statistics")

DQN Visualizer created successfully!
✓ Neural network Q-value visualization
✓ Policy extraction from learned Q-values
✓ Training progress monitoring
✓ Experience buffer statistics


### Step-by-Step DQN Training

Let's create an interactive version where you can see DQN learning step by step!

In [ ]:
class InteractiveDQN:
    def __init__(self, env):
        self.env = env
        self.agent = None
        self.visualizer = DQNVisualizer(env)
        self.current_state = None
        self.episode_step = 0
        self.create_widgets()
        
    def create_widgets(self):
        """Create interactive controls"""
        self.step_button = widgets.Button(description='➡️ Next Step')
        self.episode_button = widgets.Button(description='🎬 Full Episode')
        self.train_button = widgets.Button(description='🏋️ Train 50 Episodes')
        self.reset_button = widgets.Button(description='🔄 Reset Agent')
        self.test_button = widgets.Button(description='🧪 Test Policy')
        
        # Hyperparameter controls
        self.lr_slider = widgets.FloatLogSlider(value=0.001, base=10, min=-4, max=-1, step=0.1, description='Learning Rate:')
        self.epsilon_slider = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.01, description='Initial Epsilon:')
        self.buffer_size_slider = widgets.IntSlider(value=10000, min=1000, max=50000, step=1000, description='Buffer Size:')
        self.network_size_dropdown = widgets.Dropdown(
            options=[('Small [32, 32]', [32, 32]), ('Medium [64, 64]', [64, 64]), ('Large [128, 64, 32]', [128, 64, 32])],
            value=[64, 64],
            description='Network Size:'
        )
        
        self.step_button.on_click(lambda b: self.single_step())
        self.episode_button.on_click(lambda b: self.full_episode())
        self.train_button.on_click(lambda b: self.train_episodes())
        self.reset_button.on_click(lambda b: self.reset())
        self.test_button.on_click(lambda b: self.test_policy())
        
        self.output = widgets.Output()
        self.display_area = widgets.Output()
        
        self.controls = widgets.VBox([
            widgets.HTML("<h4>🎛️ Controls</h4>"),
            widgets.HBox([self.step_button, self.episode_button, self.train_button]),
            widgets.HBox([self.reset_button, self.test_button]),
            widgets.HTML("<h4>⚙️ Hyperparameters</h4>"),
            self.lr_slider,
            self.epsilon_slider,
            self.buffer_size_slider,
            self.network_size_dropdown
        ])
        
    def initialize_agent(self):
        """Initialize DQN agent with current hyperparameters"""
        self.agent = DQNAgent(
            self.env,
            lr=self.lr_slider.value,
            epsilon=self.epsilon_slider.value,
            buffer_size=self.buffer_size_slider.value,
            hidden_sizes=self.network_size_dropdown.value
        )
        
    def single_step(self):
        """Execute one step of DQN training"""
        with self.output:
            clear_output(wait=True)
            
            if self.agent is None:
                self.initialize_agent()
                # Ensure agent was properly initialized
                assert self.agent is not None, "Failed to initialize DQN agent"
            
            if self.current_state is None:
                # Start new episode
                self.current_state = self.env.reset()
                self.episode_step = 0
                print(f"🚀 Starting Episode {self.agent.episode_count + 1}")
                print(f"🎯 Initial state: {self.env.get_state()}")
                self.update_display()
                return
            
            # Choose action
            action = self.agent.act(self.current_state)
            action_name = Action(action).name
            exploration = "🎲 Exploration" if random.random() < self.agent.epsilon else "🎯 Neural Network"
            
            # Take action
            next_state, reward, done, _ = self.env.step(action)
            
            # Store experience
            self.agent.remember(self.current_state, action, reward, next_state, done)
            
            # Train if enough experiences
            loss = None
            if len(self.agent.replay_buffer) >= self.agent.batch_size:
                loss = self.agent.replay()
            
            # Display step information
            print(f"Step {self.episode_step + 1}:")
            print(f"  State: {self.env.get_state()} → Action: {action_name} ({exploration})")
            print(f"  Next State: {self.env.get_state()}, Reward: {reward:+.1f}")
            if loss is not None:
                print(f"  Network Loss: {loss:.4f}")
            print(f"  Buffer Size: {len(self.agent.replay_buffer)}, Epsilon: {self.agent.epsilon:.3f}")
            
            self.episode_step += 1
            self.current_state = next_state
            self.agent.total_steps += 1
            
            # Update target network periodically
            if self.agent.total_steps % self.agent.target_update == 0:
                self.agent.update_target_network()
                print("  🎯 Target network updated!")
            
            if done:
                print(f"\n✅ Episode {self.agent.episode_count + 1} completed in {self.episode_step} steps!")
                self.agent.episode_rewards.append(sum([reward]))  # This is simplified
                self.agent.episode_steps.append(self.episode_step)
                self.agent.episode_count += 1
                self.current_state = None
            
            self.update_display()
    
    def full_episode(self):
        """Run a complete episode"""
        with self.output:
            clear_output(wait=True)
            
            if self.agent is None:
                self.initialize_agent()
                # Ensure agent was properly initialized
                assert self.agent is not None, "Failed to initialize DQN agent"
            
            experiences, reward, steps = self.agent.run_episode()
            
            print(f"🎬 Episode {self.agent.episode_count} completed:")
            print(f"  Steps: {steps}")
            print(f"  Total Reward: {reward:+.1f}")
            print(f"  Average Reward per Step: {reward/steps:+.2f}")
            print(f"  Buffer Size: {len(self.agent.replay_buffer)}")
            print(f"  Current Epsilon: {self.agent.epsilon:.3f}")
            
            self.current_state = None
            self.update_display()
    
    def train_episodes(self):
        """Train for multiple episodes"""
        with self.output:
            clear_output(wait=True)
            
            if self.agent is None:
                self.initialize_agent()
                # Ensure agent was properly initialized
                assert self.agent is not None, "Failed to initialize DQN agent"
            
            print("🏋️ Training for 50 episodes...")
            start_episode = self.agent.episode_count
            
            for i in range(50):
                experiences, reward, steps = self.agent.run_episode()
                if (i + 1) % 10 == 0:
                    avg_reward = np.mean(self.agent.episode_rewards[-10:])
                    print(f"  Episode {start_episode + i + 1}: {steps} steps, {reward:+.1f} reward (Avg: {avg_reward:+.2f})")
            
            # Final statistics
            avg_reward = np.mean(self.agent.episode_rewards[-50:])
            avg_steps = np.mean(self.agent.episode_steps[-50:])
            avg_loss = np.mean(self.agent.losses[-100:]) if self.agent.losses else 0
            
            print(f"\n📊 Training Summary:")
            print(f"  Episodes: {start_episode + 1} → {self.agent.episode_count}")
            print(f"  Average Reward (last 50): {avg_reward:+.2f}")
            print(f"  Average Steps (last 50): {avg_steps:.1f}")
            print(f"  Average Loss (last 100): {avg_loss:.4f}")
            print(f"  Final Epsilon: {self.agent.epsilon:.3f}")
            
            self.current_state = None
            self.update_display()
    
    def test_policy(self):
        """Test the learned policy"""
        with self.output:
            clear_output(wait=True)
            
            if self.agent is None:
                print("❌ Please train the agent first!")
                return
            
            print("🧪 Testing learned policy (no exploration, no training)...")
            path, reward, steps = self.agent.test_policy()
            
            print(f"🏆 Policy Test Results:")
            print(f"  Path: {' → '.join(map(str, path))}")
            print(f"  Steps: {steps}")
            print(f"  Total Reward: {reward:+.1f}")
            
            if reward > 5:  # Assuming goal reward is 10, minus some step penalties
                print("✅ Excellent! Neural network learned a good policy.")
            else:
                print("📚 Policy needs more training. Try more episodes!")
    
    def reset(self):
        """Reset the agent"""
        with self.output:
            clear_output(wait=True)
            
            self.agent = None
            self.current_state = None
            self.episode_step = 0
            
            print("🔄 DQN Agent reset!")
            print("🧠 Neural networks will be reinitialized")
            print("💾 Experience buffer will be cleared")
            print("➡️ Click 'Next Step' to start learning with new hyperparameters")
            
            self.update_display()
    
    def update_display(self):
        """Update the visualization"""
        with self.display_area:
            clear_output(wait=True)
            
            if self.agent is None:
                display(widgets.HTML("<p>👆 Initialize agent by clicking 'Next Step' or other training buttons.</p>"))
                return
            
            # Create visualizations
            grid_html = self.visualizer.create_grid_html()
            q_grid = self.agent.get_q_values_grid()
            q_values_html = self.visualizer.create_q_values_html(q_grid)
            policy_html = self.visualizer.create_policy_html(q_grid)
            progress_html = self.visualizer.create_training_progress_html(self.agent)
            
            grid_widget = widgets.HTML(value=grid_html)
            q_values_widget = widgets.HTML(value=q_values_html)
            policy_widget = widgets.HTML(value=policy_html)
            progress_widget = widgets.HTML(value=progress_html)
            
            display(widgets.VBox([
                widgets.HTML(f"<h4>🤖 DQN Learning (Episode {self.agent.episode_count + 1})</h4>"),
                widgets.HBox([grid_widget, progress_widget]),
                widgets.HTML("<h4>🧠 Neural Network Q-Values and Policy</h4>"),
                widgets.VBox([
                    q_values_widget,
                    policy_widget
                ])
            ]))
    
    def start_interactive(self):
        """Start the interactive interface"""
        self.reset()
        return widgets.VBox([
            widgets.HTML("<h3>🧠 Interactive Deep Q-Network (DQN)</h3>"),
            widgets.HTML("<p><strong>DQN</strong> uses neural networks to learn Q-values and handle large state spaces!</p>"),
            widgets.HTML("<p>🔬 Key features: Experience replay, target networks, and epsilon decay.</p>"),
            self.controls,
            self.output,
            self.display_area
        ])

# Create enhanced environment and interactive DQN
env = GridWorldDQN(width=5, height=5)
interactive_dqn = InteractiveDQN(env)

print("🎮 Interactive DQN ready!")
print("Experiment with different hyperparameters and watch the neural network learn!")

# Display the interface
display(interactive_dqn.start_interactive())

🎮 Interactive DQN ready!
Experiment with different hyperparameters and watch the neural network learn!


## 🎯 Deep Q-Network Learning Summary

After exploring DQN, consider these key insights compared to previous methods:

### 🔍 **Key Advantages of DQN:**

| Aspect | Tabular Q-Learning | DQN |
|--------|-------------------|-----|
| **State Space** | Small discrete only | Can handle large/continuous |
| **Memory Usage** | O(|S| × |A|) | O(network parameters) |
| **Generalization** | None | Can generalize to unseen states |
| **Computation** | Fast lookup | Requires forward/backward pass |
| **Convergence** | Guaranteed (under conditions) | Approximation, may be unstable |

### 🧠 **Key DQN Innovations:**

1. **Experience Replay**: Breaks temporal correlations, improves sample efficiency
2. **Target Network**: Stabilizes training by providing consistent targets
3. **Function Approximation**: Enables scaling to larger problems
4. **Batch Learning**: Learns from multiple experiences simultaneously

### 🚀 **When to Use DQN vs Tabular Methods:**

- **Use Tabular Q-Learning**: Small discrete state spaces, need exact values, simple problems
- **Use DQN**: Large state spaces, continuous states, need generalization, complex environments

### 🎮 **Important Considerations:**

**DQN Challenges:**
- **Sample Efficiency**: Often needs more experiences than tabular methods
- **Hyperparameter Sensitivity**: Learning rate, network architecture, buffer size matter
- **Stability**: Can be unstable without proper techniques (target network, experience replay)
- **Approximation**: Neural network may not perfectly represent Q-function

**DQN Benefits:**
- **Scalability**: Can handle millions of states
- **Generalization**: Learns patterns that transfer to similar states
- **Flexibility**: Can incorporate different state representations
- **Modern ML**: Integrates with deep learning advances

